# 📊 Gym Inflation vs Income: Is Fitness Becoming a Luxury?

In this analysis, we explore how the cost of gym memberships has changed relative to median household income in the U.S. from 2000 to 2024. This helps us understand if staying fit is becoming financially harder — a key piece in our broader project on how aesthetics may be overtaking health in fitness culture.


In [1]:
import pandas as pd

# Load uploaded files
income_path = "../data/income_data.csv"
gym_costs_path = "../data/gym_costs.csv"

income_df = pd.read_csv(income_path)
gym_df = pd.read_csv(gym_costs_path)

income_df.head(), gym_df.head()


(  observation_date  MEHOINUSA672N
 0       2000-01-01          70020
 1       2001-01-01          68870
 2       2002-01-01          68310
 3       2003-01-01          68350
 4       2004-01-01          68250,
   Consumer Price Index for All Urban Consumers (CPI-U)  \
 0                                Original Data Value     
 1                                                NaN     
 2                                         Series Id:     
 3                            Not Seasonally Adjusted     
 4                                      Series Title:     
 
                                           Unnamed: 1 Unnamed: 2 Unnamed: 3  \
 0                                                NaN        NaN        NaN   
 1                                                NaN        NaN        NaN   
 2                                     CUUR0000SERF01        NaN        NaN   
 3                                                NaN        NaN        NaN   
 4  Club membership for shopping clubs

In [2]:
import pandas as pd

# Step 1: Dynamically find the header row in gym CPI dataset
for i, row in gym_df.iterrows():
    if 'Year' in row.tolist():
        header_row = i
        break
else:
    raise ValueError("Header row with 'Year' not found.")

# Step 2: Reload the dataset with proper header
clean_gym_df = pd.read_csv(gym_costs_path, skiprows=header_row + 1)

# Step 3: Drop rows where 'Year' is missing or not numeric
clean_gym_df = clean_gym_df[pd.to_numeric(clean_gym_df['Year'], errors='coerce').notnull()]
clean_gym_df['Year'] = clean_gym_df['Year'].astype(int)

# Step 4: Calculate average CPI from monthly columns
monthly_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Ensure all month columns are numeric
clean_gym_df[monthly_cols] = clean_gym_df[monthly_cols].apply(pd.to_numeric, errors='coerce')

# Compute annual average CPI
clean_gym_df['AvgGymCPI'] = clean_gym_df[monthly_cols].mean(axis=1)

# Step 5: Keep only 'Year' and average CPI
gym_cpi_df = clean_gym_df[['Year', 'AvgGymCPI']]

# Step 6: Clean income dataset
income_df['Year'] = pd.to_datetime(income_df['observation_date']).dt.year
income_df = income_df.rename(columns={'MEHOINUSA672N': 'MedianIncome'})
income_df = income_df[['Year', 'MedianIncome']]

# Step 7: Merge CPI and income data
merged_df = pd.merge(gym_cpi_df, income_df, on='Year')

# Step 8: Display results
print("📊 Gym CPI vs Median Income")
display(merged_df)


📊 Gym CPI vs Median Income


,Year,AvgGymCPI,MedianIncome
0,2000,108.891667,70020
1,2001,111.650000,68870
2,2002,112.941667,68310
3,2003,116.108333,68350
4,2004,116.675000,68250
5,2005,117.433333,69310
6,2006,121.850000,70080
7,2007,123.703167,71210
8,2008,125.826250,68780
9,2009,125.667500,68340


In [ ]:
# Normalize both to 2000 = 100
base_year = 2000
merged_df['GymCPI_Index'] = merged_df['AvgGymCPI'] / merged_df.loc[merged_df['Year'] == base_year, 'AvgGymCPI'].values[0] * 100
merged_df['Income_Index'] = merged_df['MedianIncome'] / merged_df.loc[merged_df['Year'] == base_year, 'MedianIncome'].values[0] * 100


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(merged_df['Year'], merged_df['GymCPI_Index'], label='Gym CPI (Indexed)', linewidth=2)
plt.plot(merged_df['Year'], merged_df['Income_Index'], label='Median Income (Indexed)', linewidth=2)
plt.axvline(x=2008, color='gray', linestyle='--', alpha=0.5, label='2008 Recession')
plt.axvline(x=2020, color='gray', linestyle='--', alpha=0.5, label='COVID-19')

plt.title('Growth of Gym Costs vs Income (2000 = 100)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Index (2000 = 100)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
merged_df['Gym_to_Income_Ratio'] = merged_df['AvgGymCPI'] / merged_df['MedianIncome']


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(merged_df['Year'], merged_df['Gym_to_Income_Ratio'], marker='o', linewidth=2)
plt.title('Gym Cost as a Proportion of Income', fontsize=14)
plt.xlabel('Year')
plt.ylabel('CPI / Income')
plt.grid(True)
plt.tight_layout()
plt.show()
